# GeoSPARQL Extension Functions (rdflib-geosparql)

The following functions are implemented in rdflib-geosparql, but are not (yet) officially standardized by OGC.

## Installing requirements

In [1]:
!pip install -r requirements.txt --break-system-packages
!pip install ipyleaflet --break-system-packages
!jupyter labextension install jupyter-leaflet

import rdflib
import shapely.geometry
from shapely import wkt
from shapely.ops import transform
from rdflib import Graph, Literal
import os
import itertools
from shapely.testing import assert_geometries_equal
from geosparql.geosparql import LiteralUtils
from geosparql.geosparql_aggregates import processLiteralTypeToGeom
from IPython.display import display, HTML
import ipywidgets as W
from ipyleaflet import Map, WKTLayer, FullScreenControl, Popup#, Tooltip

mapcenter=(34.1,-83.2)
zoomlevel=10
GEO = "http://www.opengis.net/ont/geosparql#"
GEOF = "http://www.opengis.net/def/function/geosparql/"
GEOFEXT = "http://www.opengis.net/def/function/geosparql/ext/"

def getHTMLStringFromQueryResult(qres):
    res="<table><tr><th>Variable</th><th>Value</th></tr>"
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
    for row in resultlist:
        for item in row:
            if isinstance(row[item],Literal) and row[item].datatype!=None:
                res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item].datatype)+"\">"+str(row[item]).strip()+"</a></td></tr>"
            elif isinstance(row[item],URIRef):
                res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item])+"\">"+str(row[item]).strip()+"</a></td></tr>"
            else:
                res+="<tr><td>"+str(item)+"</td><td>"+str(row[item]).strip()+"</td></tr>"
    res+="</table>"
    return res

def getMapFromWKTResult(qres,rows=[],lmap=None):
    layers=[]
    lastcentroid=shapely.geometry.Point(mapcenter[0],mapcenter[1])
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
    for row in rows:
        if row in resultlist[0]:
            popup="<ul>"
            for rowres in resultlist:
                for item in rowres:
                    if isinstance(rowres[item],Literal) and rowres[item].datatype!=None:
                        popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item].datatype)+"\">"+str(rowres[item]).strip()+"</a></li>"
                    elif isinstance(rowres[item],URIRef):
                        poup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item])+"\">"+str(rowres[item]).strip()+"</a></li>"
                    else:
                        popup+="<li><b>"+str(item)+":</b> "+str(rowres[item]).strip()+"</li>"
            popup+="</ul>"
            toprocess=resultlist[0][row].strip()
            if toprocess.startswith("<http"):
                toprocess=toprocess[toprocess.find(">")+1:]
            geom = wkt.loads(toprocess)
            if not geom.centroid.is_empty:
                lastcentroid=geom.centroid
            nlayer=WKTLayer(wkt_string=geom.wkt,hover_style={"fillColor": "red"})
            msg=W.HTML()
            msg.value=popup
            nlpopup=Popup(
                location=[lastcentroid.y,lastcentroid.x],
                child=msg,
                close_button=True,
                auto_close=False,
                close_on_escape_key=False
            )
            layers.append(nlayer)
            layers.append(nlpopup)
            nlpopup.close_popup()
    if lmap==None:
        lmap=Map(center=(lastcentroid.y,lastcentroid.x), zoom=zoomlevel)
        control = FullScreenControl()
        lmap.add(control)
    for lay in layers:
        lmap.add(lay)
    return lmap    

def processQueryResults(qres,rows=[],lmap=None):
    display(HTML(getHTMLStringFromQueryResult(qres)))
    if rows!=[]:
        lmap=getMapFromWKTResult(qres,rows,lmap)
    return lmap

g=Graph()
g.parse("tests/testdata.ttl")
print(len(g))

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
(Deprecated) Installing extensions with the jupyter labextension install command is now deprecated and will be removed in a future major version of JupyterLab.

Users should manage prebuilt extensions with package managers like pip and conda, and extension authors are encouraged to distribute their extensions as prebuilt packages 
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:54: UserWarning: An error occurred.
  warnings.warn("An error occurred.")
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:55: UserWarning: PermissionError: [Errno 13] Permission denied: '/usr/local/share/jupyter/lab/extensions/jupyter-leaflet-0.20.0.tgz'
  warnings.warn(msg[-1].strip())
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:56: UserWarning: See the log file for details: /tmp/jupyterlab-debug-pr3bvjz8.log


## GeoSPARQL Serializations

### geofext:asGeoCode Function

In [2]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gc
WHERE {
  my:A geo:hasGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGeocode(?aLiteral,"http://opengis.net/ont/geocode/OpenLocationCode") as ?gc)
}
"""),["aLiteral"],None)

POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
http://opengis.net/ont/geocode/OpenLocationCode


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gc,2G8PJ822+22


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asOBJ Function

In [6]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?obj
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asOBJ(?aLiteral) as ?obj)
}
"""),["aLiteral"],None)

GEOMTOLIT: <trimesh.Trimesh(vertices.shape=(8, 3), faces.shape=(12, 3))>
<class 'trimesh.base.Trimesh'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asJSONFG Function

In [3]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?jsonfg
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asJSONFG(?aLiteral) as ?jsonfg)
}
"""),["aLiteral"],None)

GEOMTOLIT: POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Directional relation functions

### geofext:above Function

In [4]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?abovee) as ?above) (xsd:boolean(?nabovee) as ?notabove)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:above(?aLiteral, ?dLiteral) as ?nabovee)
  BIND (geof:above(?dLiteral, ?aLiteral) as ?abovee)
}
"""),["aLiteral","dLiteral"],None)

[-83.6  34.1 -83.2  34.5]
[-83.3  33.  -83.1  33.2]
0.09999999999999432
34.5 < 33.0 = False
GEOM1 ABOVE GEOM2? False
[-83.3  33.  -83.1  33.2]
[-83.6  34.1 -83.2  34.5]
0.09999999999999432
33.2 < 34.1 = True
GEOM1 ABOVE GEOM2? True


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"
above,true
notabove,false


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:below Function

In [8]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?beloww) as ?below) (xsd:boolean(?nbeloww) as ?notbelow)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:below(?aLiteral, ?dLiteral) as ?nbeloww)
  BIND (geof:below(?dLiteral, ?aLiteral) as ?beloww)
}
"""),["aLiteral","dLiteral"],None)

[-83.6  34.1 -83.2  34.5]
[-83.3  33.  -83.1  33.2]
0.09999999999999432
34.5 < 33.0 = False
GEOM1 BELOW GEOM2? False
[-83.3  33.  -83.1  33.2]
[-83.6  34.1 -83.2  34.5]
0.09999999999999432
33.2 < 34.1 = True
GEOM1 BELOW GEOM2? True


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"
below,true
notbelow,false


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:leftOf Function

In [29]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?leftt) as ?left) (xsd:boolean(?nleftt) as ?notleft)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:leftOf(?aLiteral, ?dLiteral) as ?leftt)
  BIND (geof:leftOf(?dLiteral, ?aLiteral) as ?nleftt)
}
"""),["aLiteral","dLiteral"],None)

[-83.6  34.1 -83.2  34.5]
[-82.3  34.  -82.1  34.2]
0.10000000000000142
-83.2 < -82.3 = True
GEOM1 LEFTOF GEOM2? True
[-82.3  34.  -82.1  34.2]
[-83.6  34.1 -83.2  34.5]
0.10000000000000142
-82.1 < -83.6 = False
GEOM1 LEFTOF GEOM2? False


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"
left,true
notleft,false


Map(center=[34.099999999999994, -82.2], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…

## Non-topological query functions

### geofext:boundingDiagonal Function

In [11]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bdiag
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:boundingDiagonal(?aLiteral) as ?bdiag)
}
"""),["aLiteral","bdiag"],None)

GEOMTOLIT: LINESTRING (-83.6 -83.2, 34.1 34.5)
<class 'shapely.geometry.linestring.LineString'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bdiag,"LINESTRING (-83.6 -83.2, 34.1 34.5)"


Map(center=[-24.35, -24.749999999999993], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_…

### geofext:closestPoint Function

In [12]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?cPoint
WHERE {
  my:A my:hasPointGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:closestPoint(?aLiteral, ?dLiteral) as ?cPoint)
}
"""),["aLiteral","dLiteral","cPoint"],None)

UnboundLocalError: cannot access local variable 'resultlist' where it is not associated with a value

### geofext:compactnessRatio Function

In [14]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?cratio
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:compactnessRatio(?aLiteral) as ?cratio)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
cratio,0.886226925452758


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:endPoint Function

Returns the last point of a given geometry

In [16]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?endPoint
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:endPoint(?literal) AS ?endPoint)
}
"""),["literal","endPoint"],None)

GEOMTOLIT: POINT (-83.6 34.1)
<class 'shapely.geometry.point.Point'>


Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
endPoint,POINT (-83.6 34.1)


Map(center=[34.1, -83.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:exteriorRing Function

In [17]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?exRing
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:exteriorRing(?literal) AS ?exRing)
}
"""),["literal","exRing"],None)

GEOMTOLIT: LINEARRING (-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1)
<class 'shapely.geometry.polygon.LinearRing'>


Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
exRing,"LINEARRING (-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1)"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:farthestCoordinate Function

In [19]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?farthestCoord
WHERE {
  my:A my:hasPointGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:farthestCoordinate(?aLiteral, ?dLiteral) as ?farthestCoord)
}
"""),["aLiteral","dLiteral","farthestCoord"],None)

UnboundLocalError: cannot access local variable 'resultlist' where it is not associated with a value

### geofext:frechetDistance Function

In [21]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?fdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:frechetDistance(?cLiteral, ?fLiteral) as ?fdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,Point(-83.4 34.4)
fdistance,0.41231056256177195


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:force2D Function

In [23]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?force2D
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:force2D(?literal) AS ?force2D)
}
"""),["literal","force2D"],None)

GEOMTOLIT: POLYGON ((-77.089005 38.913574, -77.029953 38.913574, -77.029953 38.886321, -77.089005 38.886321, -77.089005 38.913574))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
literal,"Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"
force2D,"POLYGON ((-77.089005 38.913574, -77.029953 38.913574, -77.029953 38.886321, -77.089005 38.886321, -77.089005 38.913574))"


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### geof:hausdorffDistance Function

In [24]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?hdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:hausdorffDistance(?cLiteral, ?fLiteral) as ?hdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,Point(-83.4 34.4)
hdistance,0.41231056256177195


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:isClosed Function

In [25]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?e_wkt ?isAClosed ?isEClosed {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isClosed(?a_wkt) AS ?isAClosed)
    <http://example.org/ApplicationSchema#EExactGeom> geo:asWKT ?e_wkt .
    BIND(geof:isClosed(?e_wkt) AS ?isEClosed)
}
"""),["a_wkt","e_wkt"],None)

Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isAClosed,true
e_wkt,"LineString(-83.4 34.0, -83.3 34.3)"
isEClosed,false


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:isCollection Function

In [27]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?C1 ?C2 ?C3 ?isC1 ?isC2 ?isC3 {
    BIND("GEOMETRYCOLLECTION (MULTIPOINT((0 0), (1 1)), POINT(3 4), LINESTRING(2 3, 3 4))"^^geo:wktLiteral as ?C1)
    BIND("MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"^^geo:wktLiteral as ?C2)
    BIND("POLYGON ((0 0, 4 0, 4 4, 0 4, 0 0), (1 1, 1 2, 2 2, 2 1, 1 1))"^^geo:wktLiteral as ?C3)
    BIND(geof:isCollection(?C1) AS ?isC1)
    BIND(geof:isCollection(?C2) AS ?isC2)
    BIND(geof:isCollection(?C3) AS ?isC3)
}
"""),["C1","C2","C3"],None)

Variable,Value
C1,"GEOMETRYCOLLECTION (MULTIPOINT((0 0), (1 1)), POINT(3 4), LINESTRING(2 3, 3 4))"
C2,"MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"
C3,"POLYGON ((0 0, 4 0, 4 4, 0 4, 0 0), (1 1, 1 2, 2 2, 2 1, 1 1))"
isC1,true
isC2,true
isC3,false


Map(center=[2.033333333333333, 2.033333333333333], controls=(ZoomControl(options=['position', 'zoom_in_text', …

### geofext:isRing Function

In [28]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?a_wkt ?o_wkt ?isARing ?isORing {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isRing(?a_wkt) AS ?isARing)
            <http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
    BIND(geof:isRing("POINT M (1 2 3)"^^geo:wktLiteral) AS ?isORing)
}
"""),["a_wkt","o_wkt"],None)

Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isARing,true
o_wkt,"Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
isORing,false


Map(center=[47.47714370265989, 2.4756124648711317], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:longestLine Function

In [30]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?aLiteral ?dLiteral ?longestLine
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:longestLine(?aLiteral, ?dLiteral) as ?longestLine)
}
"""),["aLiteral","dLiteral","longestLine"],None)

LONGESTLINE
0.31622776601683567
0.5099019513592787
0.5099019513592787
0.31622776601683567
0.31622776601683567
0.1414213562373065
0.14142135623731653
0.14142135623731653
0.1414213562373065
0.1414213562373065
0.5099019513592773
0.5099019513592802
0.31622776601683794
0.31622776601683344
0.5099019513592773
0.5830951894845285
0.7071067811865476
0.5830951894845285
0.42426406871192446
0.5830951894845285
0.31622776601683567
0.5099019513592787
0.5099019513592787
0.31622776601683567
0.31622776601683567
GEOMTOLIT: LINESTRING (-83.6 34.5, -83.1 34)
<class 'shapely.geometry.linestring.LineString'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
longestLine,"LINESTRING (-83.6 34.5, -83.1 34)"


Map(center=[34.25, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…